# Chapter 8 — CNN Geometry

**Book alignment:** PyTorch From First Principles, Chapter 8

**Question this notebook isolates:** Can we predict a convolution's output shape (channels declared, spatial extent derived) before running — and does the `[B, H, W, C]` vs `[B, C, H, W]` mismatch explain the `224 channels` error exactly?


In [ ]:
import numpy as np
import torch
import torch.nn as nn

torch.manual_seed(0)
np.random.seed(0)


## 1. One convolution, computed by hand

4×4 input, 2×2 diagonal kernel `[[1, 0], [0, 1]]`, bias `0.5`: output `[i, j] = inp[i, j] + inp[i+1, j+1] + 0.5`. PyTorch must match the hand table exactly.


In [ ]:
inp = torch.tensor([[[[1., 2., 3., 4.],
                      [5., 6., 7., 8.],
                      [9., 10., 11., 12.],
                      [13., 14., 15., 16.]]]])
conv = nn.Conv2d(1, 1, kernel_size=2, bias=True)
with torch.no_grad():
    conv.weight.copy_(torch.tensor([[[[1., 0.], [0., 1.]]]]))
    conv.bias.copy_(torch.tensor([0.5]))
out = conv(inp)
expected = torch.tensor([[[[7.5, 9.5, 11.5],
                           [15.5, 17.5, 19.5],
                           [23.5, 25.5, 27.5]]]])
print(f"out shape={tuple(out.shape)}")
print(out.squeeze().tolist())


In [ ]:
assert tuple(out.shape) == (1, 1, 3, 3)
assert torch.allclose(out, expected), out
print("mechanism confirmed: sliding dot products plus bias")


## 2. The spatial formula predicts every case before execution

`out = floor((W + 2P - D(K-1) - 1) / S + 1)`: kernel-3 valid → 6, stride-2 pad-1 → 4, same-pad → 8, maxpool-2 → 3.


In [ ]:
def out_size(w, k, s=1, p=0, d=1):
    return (w + 2 * p - d * (k - 1) - 1) // s + 1

cases = [
    (dict(out_channels=5, kernel_size=3), (2, 3, 8, 8), (2, 5, 6, 6)),
    (dict(out_channels=5, kernel_size=3, stride=2, padding=1), (2, 3, 8, 8), (2, 5, 4, 4)),
    (dict(out_channels=5, kernel_size=3, padding=1), (2, 3, 8, 8), (2, 3 + 2, 8, 8)),
]
for kwargs, in_shape, want in cases:
    c = nn.Conv2d(in_channels=in_shape[1], **kwargs)
    got = tuple(c(torch.randn(*in_shape)).shape)
    pred = (in_shape[0], kwargs["out_channels"], out_size(in_shape[2], 3, kwargs.get("stride", 1), kwargs.get("padding", 0)), out_size(in_shape[3], 3, kwargs.get("stride", 1), kwargs.get("padding", 0)))
    print(f"{kwargs}: predicted={pred} observed={got}")
    assert got == want == pred, (got, want, pred)
pool = nn.MaxPool2d(2)
got = tuple(pool(torch.randn(2, 5, 6, 6)).shape)
print(f"maxpool2: observed={got} predicted={(2, 5, 3, 3)}")


In [ ]:
assert got == (2, 5, 3, 3)
assert out_size(224, 3) == 222
print("channels are declared; spatial extents are derived — no surprises")


## 3. `[B, H, W, C]` explains the `224 channels` error; `permute` is the repair

A `(2, 8, 8, 3)` batch into `Conv2d(3, 4, 3)` must fail naming 8 channels (axis 1); permuting to `(2, 3, 8, 8)` must yield `(2, 4, 6, 6)` with a 132-parameter layer.


In [ ]:
conv = nn.Conv2d(in_channels=3, out_channels=4, kernel_size=3)
x_hwc = torch.randn(2, 8, 8, 3)
try:
    conv(x_hwc)
    msg, raised = "", False
except RuntimeError as e:
    msg, raised = str(e), True
print(f"raised={raised} msg={msg[:80]}...")
x = x_hwc.permute(0, 3, 1, 2)
out = conv(x)
print(f"permuted={tuple(x.shape)} out={tuple(out.shape)} weight={tuple(conv.weight.shape)} params={conv.weight.numel() + conv.bias.numel()}")


In [ ]:
assert raised and "8 channels" in msg, msg
assert tuple(x.shape) == (2, 3, 8, 8)
assert tuple(out.shape) == (2, 4, 6, 6)
assert tuple(conv.weight.shape) == (4, 3, 3, 3)
assert conv.weight.numel() + conv.bias.numel() == 4 * 3 * 3 * 3 + 4 == 112
print("do not repair the dimension until you know what it means")


## What we earned

CNN geometry is mechanical: `C_out` is declared by the layer, `H_out`/`W_out` follow the kernel/stride/padding formula, and the channel axis sits at position 1. Derive first, run second, and locate the first layer where prediction and observation diverge.

Chapter 9 builds on this geometry with the training dynamics of deeper networks.
